# Decision Trees: Classification, Regression & a From-Scratch C4.5 Implementation

This project covers decision tree classification and regression using scikit-learn, including handling missing data, n-fold cross-validation, tree visualization and feature importance, overfit-avoidance parameters, cost-complexity pruning, and a real-world regression problem. It concludes with a decision tree classifier implemented from scratch using the C4.5 algorithm, benchmarked against scikit-learn's built-in implementation.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score 

## Baseline Model (Iris)

A `DecisionTreeClassifier` with default parameters, compared against one restricted to `max_depth=3`, on the Iris dataset.

In [ ]:
# Debug
# Learn the data
train_data, train_meta = arff.loadarff("data/iris.arff")
iris_df= pd.DataFrame(train_data)

# Decode "class" from bytecodes to strings!
iris_df["class"] = iris_df["class"].str.decode("utf-8")

# Quick EDA
print("EDA:")
print("\tFirst 5 rows of iris data set: ")
display(iris_df.head())

print("\n\tIris Class Distribution:")
display(iris_df["class"])

# Split into X and y
X = iris_df.drop(["class"], axis = 1)
y = iris_df["class"]

# Get data splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Initialize, fit Decision Tree with default parameters
dtc = DecisionTreeClassifier()
dtc.fit(X_train, np.ravel(y_train))

# Predictions!
y_train_pred = dtc.predict(X_train)
y_test_pred = dtc.predict(X_test)

# Accuracy!
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Train Accuracy:")
print(f"\t{train_accuracy}")
print("Test Accuracy: ")
print(f"\t{test_accuracy}")


In [ ]:
# Evaluation
# Initialize, fit Decision Tree with max_depth = 3
dtc2 = DecisionTreeClassifier(max_depth = 3)
dtc2.fit(X_train, np.ravel(y_train))

# Predictions!
y_train_pred = dtc2.predict(X_train)
y_test_pred = dtc2.predict(X_test)

# Accuracy!
train_accuracy2 = accuracy_score(y_train, y_train_pred)
test_accuracy2 = accuracy_score(y_test, y_test_pred)

print("Accuracies of Decision Tree Classifier with max_depth = 3:")
print("\tTrain Accuracy:")
print(f"\t\t{train_accuracy2}")
print("\tTest Accuracy: ")
print(f"\t\t{test_accuracy2}")

**Results**

The default-parameter tree reached both train and test accuracy of 1.0 — with no depth limit, it was able to fully memorize the training data, and in this case the resulting decision boundary generalized perfectly to the test set.

The `max_depth=3` tree had a slightly lower training accuracy (0.958), since fewer splits meant it couldn't perfectly capture every training point. Test accuracy stayed at 1.0, though — meaning the true decision boundary for this dataset is simple enough that even the shallower tree captured it correctly.

## Handling Missing Values (Voting Dataset)

The voting dataset contains missing values, encoded as `?`. This section replaces them with an explicit "Unknown" category (rather than dropping them), one-hot encodes all nominal features, and induces the tree fully (no stopping criteria) on an 80/20 split.

In [ ]:
import missingno as msno

# Learn data with missing values
train_data, train_meta = arff.loadarff("data/voting_with_missing.arff")
voting_df = pd.DataFrame(train_data)

# Quick EDA 
print("EDA:")
print("\tFirst 5 rows of voting data set: ")
display(voting_df.head())

print("\n\tVoting Class Distribution:")
display(voting_df["Class"])
print("The \"Class\" column is encoded. Decoding!")

print("\nData (without identifying missing values): ")
msno.matrix(voting_df)
plt.show()

print("Data (with identified missing values): ")
voting_na_df = voting_df.replace(b'?', np.nan)
msno.matrix(voting_na_df)
plt.show()

# Decode
voting_na_df["Class"] = voting_na_df["Class"].str.decode("utf-8")

# Get X and y
X = voting_na_df.drop(["Class"], axis = 1)
y = voting_na_df["Class"]

# Fill NA values with a new feature value to enable a split on these values:
X = X.fillna("Unknown") 

# One-hot encoding for nominal features:
X = pd.get_dummies(X)

# Create a split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)

# Model initialize and fit
dtc_voting = DecisionTreeClassifier()
dtc_voting.fit(X_train, np.ravel(y_train))

# Predictions!
y_train_pred = dtc_voting.predict(X_train)
y_test_pred = dtc_voting.predict(X_test)

# Accuracy Scores!
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Accuracies of Decision Tree Classifier with default parameters:")
print("\tTrain Accuracy:")
print(f"\t\t{train_accuracy}")
print("\tTest Accuracy: ")
print(f"\t\t{test_accuracy}")


**Results**

Missing values were first visualized (via `missingno`) to confirm their extent, then replaced with `NaN` and filled with the literal value `"Unknown"` before one-hot encoding. Treating missing values this way — rather than dropping them — lets the tree treat "unknown" as a real, informative category it can split on, rather than losing that information entirely.

Training accuracy reached 1.0, as expected for a fully induced tree with no stopping criteria — it continues splitting until every training point is classified correctly. Test accuracy came out around 94%, indicating the learned splits generalized well beyond memorizing the training set.

## N-Fold Cross-Validation (Cars Dataset)

10-fold and 5-fold cross-validation accuracies for a decision tree on the Cars dataset.

In [ ]:
from sklearn.model_selection import cross_val_score

# Learn the data, make a data frame
train_data, meta_data = arff.loadarff("data/cars.arff")
cars_df = pd.DataFrame(train_data)

# Quick EDA
print("EDA: ")
print("\tFirst 5 rows of cars data set:")
display(cars_df.head())

print("\nCar Type Distributions: ")
display(cars_df["class"].value_counts().sort_index())

print(f"Data shape: {cars_df.shape}")

# Decode all columns, one-hot encode X
print("\nThe data is in bytes! Decoding now...\n")
for col in cars_df.columns:
    cars_df[col] = cars_df[col].str.decode("utf-8")

    # Create X and y frames
X = cars_df.drop(["class"], axis = 1)
y = cars_df["class"]

X = pd.get_dummies(X)

# Do a 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 11
)

# Initialize and fit the model
dtc_cars = DecisionTreeClassifier()
dtc_cars.fit(X_train, np.ravel(y_train))

print("Cross Validation Accuracies for a 10-fold Cross Validation:")
# Calculate the 10-fold cross validation accuracies
ten_fold_cv = cross_val_score(
    dtc_cars, X, y, cv = 10, scoring = "accuracy"
)
# Calculate average
avg_ten_fold = np.mean(ten_fold_cv)
# Create and display table
ten_fold_table= pd.DataFrame({"Fold": range(1,11), "Accuracy": ten_fold_cv})
ten_fold_table.loc["Average"] = ["Average", avg_ten_fold]
ten_fold_table.set_index("Fold", inplace = True)
display(ten_fold_table)

print("\nCross Validation Accuracies for a 5-fold Cross Validation:")
# Calculate the 5-fold cross validatoin accuracies 
five_fold_cv = cross_val_score(
    dtc_cars, X, y, cv = 5, scoring = "accuracy"
)
# Calculate average
avg_five_fold = np.mean(five_fold_cv)
# Create and display table
five_fold_table = pd.DataFrame({"Fold": range(1,6), "Accuracy": five_fold_cv})
five_fold_table.loc["Average"] = ["Average", avg_five_fold]
five_fold_table.set_index("Fold", inplace = True)
display(five_fold_table)

**What cross-validation shows**

N-fold cross-validation splits the data into *n* equal folds, trains on all but one fold, and validates on the held-out fold — repeating this for every fold and averaging the results. A single train/test split can be "lucky" or "unlucky" purely by chance; cross-validation reduces that dependency by ensuring every point is used for both training (n−1 times) and testing (once), producing a more reliable accuracy estimate. It's similar in spirit to increasing sample size in a statistical study — more, varied evaluations make the resulting metric more trustworthy.

Cross-validation also reveals model stability: consistent scores across folds suggest a stable, well-generalizing model, while widely varying scores suggest sensitivity to the specific train/test split (or overfitting). In this case, the 10-fold results ranged from 0.67 to 0.92 — a fairly wide spread, suggesting the model's performance is somewhat split-dependent, with similar variation showing up in the 5-fold results.

Cross-validation itself doesn't produce a final trained model — its purpose is evaluating stability and, typically, guiding hyperparameter selection before training the model that's actually deployed.

## Visualizing the Trees and Feature Importance

Full and depth-limited (`max_depth=2`) trees for both the Voting and Cars datasets, with feature importances for each.

In [ ]:
#Print induced trees for the voting and car data sets
from sklearn import tree

# Re-fit classifiers with a max depth
    #Voting
X = voting_na_df.drop(["Class"], axis = 1)
y = voting_na_df["Class"]
X = X.fillna("Unknown")
X = pd.get_dummies(X)

voting_attributes = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)

small_dtc_voting = DecisionTreeClassifier(max_depth = 2)
small_dtc_voting.fit(X_train, np.ravel(y_train))

    #Cars
X = cars_df.drop(["class"], axis = 1)
y = cars_df["class"]
X = pd.get_dummies(X)

cars_attributes = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)

small_dtc_cars = DecisionTreeClassifier(max_depth = 2)
small_dtc_cars.fit(X_train, np.ravel(y_train))

print("Printing full trees for \"voting.arff\" and \"cars.arff\":")

plt.figure(figsize = (24,16))
tree.plot_tree(dtc_voting, feature_names = voting_attributes, filled = True)
plt.title("Full Voting Decision Tree", fontsize=20)
plt.show()

plt.figure(figsize = (24,16))
tree.plot_tree(dtc_cars, feature_names = cars_attributes, filled = True)
plt.title("Full Cars Decision Tree", fontsize=20)
plt.show()

#Print the small Decision Trees
print("Printing small (max_depth = 2) trees for \"voting.arff\" and \"cars.arff\":")

plt.figure(figsize = (12,8))
tree.plot_tree(small_dtc_voting, feature_names = voting_attributes, filled = True)
plt.title("Small Voting Decision Tree (max_depth=2)", fontsize=20)
plt.show()

plt.figure(figsize = (12,8))
tree.plot_tree(small_dtc_cars, feature_names = cars_attributes, filled = True)
plt.title("Small Cars Decision Tree (max_depth=2)", fontsize=20)
plt.show()

# Feature Importances (small, depth-limited trees)
print("\nVoting Feature Importances:")
voting_importance_df = pd.DataFrame({
    "Feature": voting_attributes,
    "Importance": small_dtc_voting.feature_importances_
})
display(voting_importance_df)

print("\nCar Feature Importances:")
cars_importance_df = pd.DataFrame({
    "Feature": cars_attributes,
    "Importance": small_dtc_cars.feature_importances_
})
display(cars_importance_df)


**Results**

The full, unrestricted trees for both datasets were large and hard to interpret visually — too dense to trace by eye. The depth-limited (`max_depth=2`) trees were far more legible and form the basis for the discussion below.

**Voting dataset:** Given the political nature of the features, `physician-fee-freeze` and `immigration` seemed like plausible candidates for important splits going in. The tree confirmed this — it splits first on `physician-fee-freeze`, then on immigration status, then on `synfuels-corporation-cutback`. Feature importances back this up dramatically: `physician-fee-freeze` alone carries an importance of 0.95, with the other two splitting features each under 0.1.

**Cars dataset:** Safety seemed like the most likely important feature going in, given its general importance to buyers. The tree splits first on low safety, then on number of persons, then acceptability. `safety_low` had an importance of 0.41 — high, though `persons` actually had a *higher* importance despite not being the first split, which was a genuinely interesting result worth digging into further. Feature importances stayed unchanged between the full and depth-limited trees for both datasets.

## Split Criterion Comparison: Gini vs. Entropy vs. Log-Loss

Decision trees on the Voting dataset (`max_depth=6`) trained with each of scikit-learn's three split criteria, comparing accuracy, tree structure, and feature importance.

In [ ]:
# Experiment with criterion parameter for voting data set 
X = voting_na_df.drop(["Class"], axis = 1)
y = voting_na_df["Class"]
X = X.fillna("Unknown")
X = pd.get_dummies(X)

voting_attributes = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)

classifier_names = ["Gini", "Entropy", "Log-loss"]

print("Decision Trees For Each Criterion:")
# Gini
gini_dtc = DecisionTreeClassifier(max_depth = 6)
gini_dtc.fit(X_train, np.ravel(y_train))

plt.figure(figsize = (12,8))
tree.plot_tree(gini_dtc, feature_names = voting_attributes, filled = True)
plt.title("Small Voting Decision Tree (Gini)", fontsize=20)
plt.show()

# Entropy
entropy_dtc = DecisionTreeClassifier(criterion = "entropy", max_depth = 6)
entropy_dtc.fit(X_train, np.ravel(y_train))

plt.figure(figsize = (12,8))
tree.plot_tree(entropy_dtc, feature_names = voting_attributes, filled = True)
plt.title("Small Voting Decision Tree (Entropy)", fontsize=20)
plt.show()

# Log-loss
logloss_dtc = DecisionTreeClassifier(criterion = "log_loss", max_depth = 6)
logloss_dtc.fit(X_train, np.ravel(y_train))

plt.figure(figsize = (12,8))
tree.plot_tree(logloss_dtc, feature_names = voting_attributes, filled = True)
plt.title("Small Voting Decision Tree (Log Loss)", fontsize=20)
plt.show()

#########################################################################

classifiers = [gini_dtc, entropy_dtc, logloss_dtc]

# Create dataframes of accuracies and feature importances!
clfs = zip(classifier_names, classifiers)
accs = []
feat_importances = {}

for name, classifier in clfs:
    # Predictions 
    train_pred = classifier.predict(X_train)
    test_pred = classifier.predict(X_test)
    
    # Accuracies
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    
    # Feature importances
    feature_importances = classifier.feature_importances_
    
    accs.append({
        "Criterion": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
    })

    feat_importances[f"{name} Importance"] = classifier.feature_importances_

acc_df = pd.DataFrame(accs)
acc_df.set_index("Criterion", inplace = True)

feat_importances = pd.DataFrame(feat_importances, index=X_train.columns)

print("\nTrain and Test Accuracies for Each Criterion:")
display(acc_df)

print("Feature Importances for Each Criterion:")
display(feat_importances)

**Results**

At shallow depth, test and train accuracy were identical across all three criteria — not the most informative comparison, since shallow trees can mask differences that would show up with more depth. Testing at greater depth did reveal cases where one criterion outperformed the others slightly.

Even where accuracy matched, the actual splits often diverged — the root node would sometimes match, but child nodes frequently differed. Feature importances varied more noticeably: differences of 0.2 or more between criteria weren't unusual, and some features scored 0.0 importance under one criterion while scoring 0.13+ under another.

The takeaway: on real-world data with deeper trees, split criterion likely has a modest effect on raw accuracy but can meaningfully change tree structure and which features get emphasized — worth considering when interpretability matters, not just predictive performance.

## Overfit Avoidance: Tree-Complexity Parameters (Cars Dataset)

A fully induced tree on the Cars dataset compared against six parameters that constrain tree growth: `min_samples_leaf`, `min_samples_split`, `min_impurity_decrease` (which allow the tree to stop growing based on information content), and `max_depth`, `max_leaf_nodes`, `max_features` (which impose hard structural limits and risk underfitting).

In [ ]:
# Explore different overfit parameters

# The Same Quick EDA - because I like to have it!
print("EDA: ")
print("\tFirst 5 rows of cars data set:")
display(cars_df.head())

print("\nCar Type Distributions: ")
display(cars_df["class"].value_counts().sort_index())

print(f"Data shape: {cars_df.shape}")

# Create X and y frames
X = cars_df.drop(["class"], axis = 1)
y = cars_df["class"]

X = pd.get_dummies(X)

# Do a 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 11
)

# Initiate and Fit Classifier, full tree, default parameters
dtc_cars = DecisionTreeClassifier()
dtc_cars.fit(X_train, np.ravel(y_train))

# Display the Full tree
cars_attributes = X.columns
plt.figure(figsize = (24,16))
tree.plot_tree(dtc_cars, feature_names = cars_attributes, filled = True)
plt.title("Full Cars Decision Tree", fontsize=20)
plt.show()

# Predictions and Accuracy Scores For Full Tree
y_train_pred = dtc_cars.predict(X_train)
y_test_pred = dtc_cars.predict(X_test)

full_train_accuracy = accuracy_score(y_train, y_train_pred)
full_test_accuracy = accuracy_score(y_test, y_test_pred)

accuracies = {"Metric": ["Train Accuracy", "Test Accuracy"], "Full Tree Value": [train_accuracy, test_accuracy]}
accuracies = pd.DataFrame(accuracies)
accuracies.set_index("Metric", inplace = True)
display(accuracies)

#Total nodes and max tree depth
print(f"Total Number of Nodes: {dtc_cars.tree_.node_count}")
print(f"Maximum Tree Depth: {dtc_cars.tree_.max_depth}")

#Function to compare the values with different values
def train_test_accuracies(classifiers, param):
    """Given a list of classifiers, fit the classifier, and train and test values, return a data frame comparing the newly
    calculated accuracy values compared to the accuracy values from the full tree."""

    accuracies = {"Metric": ["Train Accuracy", "Test Accuracy"], "Full Value": [full_train_accuracy, full_test_accuracy]}

    for classifier, param_value in classifiers:
        classifier.fit(X_train, y_train)

        y_train_pred = classifier.predict(X_train)
        y_test_pred = classifier.predict(X_test)

        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)

        accuracies[f"{param} = {param_value}"] = [train_accuracy, test_accuracy]

    accuracies = pd.DataFrame(accuracies)
    accuracies.set_index("Metric", inplace = True)

    print(f"\nAccuracy Values of a Decision Tree With Changed {param} Compared to the Full Tree:")
    display(accuracies)

# Create lists of classifiers with differing values
min_samples_leaf = []
min_samples_split = []
min_impurity_decrease = []
max_depth = []
max_leaf_nodes = []
max_features = []
for i in range(1, 8, 2):
    min_samples_leaf.append((DecisionTreeClassifier(min_samples_leaf = i), i))
    min_samples_split.append((DecisionTreeClassifier(min_samples_split = i+1), i+1)) #must be an integer >= 2, so just add one to each int
    min_impurity_decrease.append((DecisionTreeClassifier(min_impurity_decrease = i/1000), i/1000)) #must be a decimal so divide by 1000
    max_depth.append((DecisionTreeClassifier(max_depth= i), i))
    max_leaf_nodes.append((DecisionTreeClassifier(max_leaf_nodes = i + 1), i + 1)) #must be an integer >=2, so just add one to each int
    max_features.append((DecisionTreeClassifier(max_features = i), i)) 

# Display all the tables!
train_test_accuracies(min_samples_leaf, "min_samples_leaf")
train_test_accuracies(min_samples_split, "min_samples_split")
train_test_accuracies(min_impurity_decrease, "min_impurity_decrease")
train_test_accuracies(max_depth, "max_depth")
train_test_accuracies(max_leaf_nodes, "max_leaf_nodes")
train_test_accuracies(max_features, "max_features")





**Results**

The fully induced tree reached 1.0 training accuracy and about 0.976 test accuracy — expected, since an unconstrained tree can perfectly memorize training data while still generalizing reasonably well on relatively simple data like this.

**`min_samples_leaf`** (values 1–7): increasing this value requires more supporting data per leaf, reducing overfitting and smoothing the decision boundary. At the default (1), results matched the full tree exactly; accuracy dipped only slightly as the value increased, and at 7 the test accuracy was actually higher than at 5.

**`min_samples_split`** (values 2–8): similar effect — requiring more instances to justify a split reduces small, overfit-prone splits. Accuracy stayed above 0.95 across the full range tested.

**`min_impurity_decrease`** (values 0.001–0.007): even small values here had an outsized effect — at 0.007, test accuracy dropped to 0.89. This parameter reduces overfitting effectively but starts to underfit quickly as it increases.

**`max_depth`** (values 1–7): a hard depth limit severely affects both structure and accuracy — a depth of 1 achieved just 0.71 test accuracy, a clear underfit. Accuracy improved as depth increased, reaching about 0.91 at depth 7. Compared to the information-based parameters above, this carries more underfitting risk for the same accuracy target.

**`max_leaf_nodes`** (values 2–8): similarly prone to underfitting — 2 leaf nodes limited both train and test accuracy to around 0.70, improving to about 0.85 by 8 nodes, still below the information-based parameters' results.

**`max_features`** (values 1–7): restricting the features considered per split kept training accuracy at 1.0 throughout (since depth remains unconstrained, the tree can still memorize training data regardless of which features it sees), but test accuracy varied more, since it depends heavily on which features happened to be available at each split.

Overall, `min_samples_leaf`, `min_samples_split`, and `min_impurity_decrease` were the most effective at reducing overfitting without a hard structural ceiling — each lets the tree grow as needed, constrained only by how much a split is actually justified by the data, which tends to generalize better than an arbitrary depth or node-count limit.

## Cost-Complexity Pruning

A fully induced tree on the Cars dataset, pruned post-hoc using the `ccp_alpha` parameter across 7 values, comparing accuracy, node count, and depth against the unpruned baseline.

In [ ]:
# Pruning
# Create X and y frames
X = cars_df.drop(["class"], axis = 1)
y = cars_df["class"]

X = pd.get_dummies(X)

# Do a 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 11
)

# Generate ccp_alpha parameter values
ccp_alphas = [0, 0.001] + [i/1000 for i in range(5, 30, 5)]
print(ccp_alphas)

df = {"Metric": ["Train Accuracy", "Test Accuracy"]}
df2 = {"Tree Details" : ["Nodes", "Depth"]}

# Initiate and Fit Classifiers, add values to dictionary
for alpha in ccp_alphas:
    dtc_cars = DecisionTreeClassifier(ccp_alpha = alpha)
    dtc_cars.fit(X_train, np.ravel(y_train))

    y_train_pred = dtc_cars.predict(X_train)
    y_test_pred = dtc_cars.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    nodes = dtc_cars.tree_.node_count
    depth = dtc_cars.tree_.max_depth

    df[f"cpp_alpha = {alpha}"] = [train_acc, test_acc]
    df2[f"cpp_alpha = {alpha}"] = [nodes, depth]

df = pd.DataFrame(df)
df.set_index("Metric", inplace = True)
df2 = pd.DataFrame(df2)
df2.set_index("Tree Details", inplace = True)

print("\nTrain and Test Accuracies:")
display(df)
print("\nTree Nodes and Depth: ")
display(df2)


**How `ccp_alpha` works**

Cost-complexity pruning first grows the full tree, then works backward through the nodes, removing (pruning) any node that doesn't improve overall impurity by at least `ccp_alpha`. This is a form of *post*-pruning, unlike the parameters in the previous section, which are all forms of *pre*-pruning — stopping tree growth as it happens. Pre-pruning is faster but can discard a split that would have led to a useful split later on, since it never explores that branch. Post-pruning takes longer (the full tree is grown first) but avoids that risk, since every branch is explored before any pruning decision is made.

**Results**

The unpruned baseline had 191 nodes, a depth of 14, 1.0 training accuracy, and 0.97 test accuracy. As `ccp_alpha` increased across the 7 tested values (0, 0.001, 0.005, 0.010, 0.015, 0.020, 0.025), both accuracy and tree complexity decreased steadily — expected, since a larger alpha prunes more aggressively.

Two results stood out: between `ccp_alpha = 0.02` and `0.025` (both fairly aggressive, reducing nodes and depth to under 10), test accuracy was identical — suggesting that once pruning is this aggressive, further pruning may cost little to nothing in accuracy. And between `ccp_alpha = 0.001` and `0.005`, test accuracy dropped by only 0.02, while tree complexity dropped dramatically — from 117 nodes/depth 11 to just 39 nodes/depth 8. That tradeoff — a small accuracy cost for a large reduction in model complexity and computation — is a clear demonstration of why pruning is worth tuning in practice.

## Decision Tree Regression (Car Price Prediction)

A `DecisionTreeRegressor` applied to a car price dataset sourced from Kaggle, chosen for its mix of missing values, nominal and continuous features, and a continuous regression target (price).

In [ ]:
#Learn regression data set
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Read the CSV file into a data frame
car_price_df = pd.read_csv("data/car_price.csv")

# Quick EDA
print("EDA:")
print("\tFirst 5 rows of car price data set: ")
display(car_price_df.head())

# Remove the missing rows
car_price_df = car_price_df.dropna(how="all")

# Get X and y, drop Car ID
X = car_price_df.drop(["Price", "Car ID"], axis = 1)
y = car_price_df["Price"]

# One-hot Encode Categorical columns of X
categorical_cols = X.select_dtypes(include = ["object"]).columns
X = pd.get_dummies(X, columns = categorical_cols)

# Train and Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 3
)

# Initialize and Fit the Regression (with default params)
dtr = DecisionTreeRegressor(min_samples_leaf = 75, random_state = 3)
dtr.fit(X_train, np.ravel(y_train))

# Tree Statistics
total_nodes = dtr.tree_.node_count
leaf_nodes = dtr.get_n_leaves()
max_depth = dtr.tree_.max_depth

print(f"Total Nodes: {total_nodes}")
print(f"Leaf Nodes: {leaf_nodes}")
print(f"Max Tree Depth: {max_depth}")

# Prediction
y_pred = dtr.predict(X_test)

# MAE
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

# Coef. of Determination
r2 = r2_score(y_test, y_pred)
print(f"Coefficient of Determination: {r2}")


**Choosing the dataset and tuning the model**

This dataset spans an unusually wide price range — from roughly $20,000 cars up to quarter-million-dollar luxury vehicles. With default parameters, the resulting tree was extremely complex (3,599 nodes, 1,800 leaf nodes) and performed poorly: MAE around $32,600, and an R² of -1.1, meaning the model predicted worse than simply guessing the mean price every time.

Some tuning was explored to see whether this could be improved. `ccp_alpha` had no meaningful effect (if anything, slightly worse). `min_samples_split` similarly made little difference. `min_samples_leaf` was the most effective lever tested — at a value of 75, MAE dropped to around $24,200 and R² improved to about -0.05 (still below baseline-guess performance, but much closer), while the tree shrank to just 33 nodes and a max depth of 6.

The wide price range in this dataset is likely the core issue — a single feature set struggles to capture both economy and luxury car pricing dynamics well. A more targeted approach (e.g. separate models per price tier, or log-transforming price) would likely perform meaningfully better than further hyperparameter tuning alone.

## A From-Scratch C4.5 Decision Tree Classifier

A C4.5-style decision tree classifier, implemented from scratch using gain ratio (rather than CART's plain information gain) as the split criterion, with support for unknown/missing attribute values. Benchmarked against scikit-learn's `DecisionTreeClassifier` on both a small hand-checkable dataset and the full Voting dataset.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn import datasets

class Node: # A node class to help with the tree construction
    def __init__(self, feature = None, children = None, *, value=None, majority = None):
        self.feature = feature
        self.children = children if children is not None else {}
        self.value = value
        self.majority = majority # majority class 
    
    def is_leaf_node(self):
        return self.value is not None

class DTClassifier(BaseEstimator,ClassifierMixin):

    def __init__(self):
        """Initialize the classifier. The tree is built lazily in fit()."""
        self.root = None
    
    def majority_label(self, y):
        """Find the most common label in y."""
        # Create a counts dictionary, find how many times each label occurs
        counts = {}
        for label in y:
            if label in counts:
                counts[label] += 1
            else:
                counts[label] = 1
        
        # Find label with max count from dictionary
        max_count = -1
        majority_label = None
        for label, count in counts.items(): # Not sure how to do this more efficiently... they should let you sort a dictionary by values :(
            if count > max_count:
                max_count = count
                majority_label = label
        return majority_label
    
    def entropy(self, y):
        """A helper function that Calculates entropy for the y values."""
        _, counts = np.unique(y, return_counts = True) #we don't need values here, so stored in an underscore
        p_values = counts / len(y)

        entropy = 0
        for p in p_values:
            if p > 0:
                entropy -= p * np.log2(p) # where entropy is the negative sum of the probability times the log base 2 of the probability 
        
        return entropy


    def gain_ratio(self, X_column, y):
        """Calculate gain ratio for a given split."""
        # Parent Entropy
        parent_entropy = self.entropy(y) #using the entropy function thatt was written 

        values, counts = np.unique(X_column, return_counts = True) #get unique values (and counts of each!)

        split_info = 0
        weighted_entropy = 0

        for value, count in zip(values, counts): #iterate through all possible splits
            small_y = y[X_column == value] # get the number of y's that equal our value
            probability = count / len(y) # calculate the probability

            weighted_entropy += probability * self.entropy(small_y) #calculate weighted entropy, expected entropy if the info was split by this feature

            if probability > 0:
                split_info -= probability * np.log2(probability) #features with more unique values have lower gain ratios... (they provide less info!)
        
        info_gain = parent_entropy - weighted_entropy #just normal information gain

        if split_info == 0: 
            return 0
        
        return info_gain / split_info #gain ratio is the information gain divided by the split information

    def find_best_feature(self, X, y):
        """Given the X and y, calculate the best feature to split on based on the gain ratio."""
        best_gain = -1
        best_feature = None

        for feature_idx in range(X.shape[1]):
            X_column = X[:, feature_idx]
            gain = self.gain_ratio(X_column, y) #calculate the gain ratio for each

            if gain > best_gain: #check if it is better than the best
                best_gain = gain
                best_feature = feature_idx
        
        return best_feature, best_gain # return the best ones!

    def grow_tree(self, X, y, depth = 0):
        """A helper function that grows the tree according to what feature will result in the best split,
        creates nodes, returns resulting Node from calculations."""
        majority = self.majority_label(y) #calculate the majority from what is in y

        if len(np.unique(y)) == 1: #If there is only one more feature in y, return a node with the remaining feature as the value
            return Node(value=y[0], majority = majority)

        if X.shape[1] == 0: 
            return Node(value = majority, majority = majority)

        best_feature, best_gain = self.find_best_feature(X, y)

        if best_gain == 0:
            return Node(value = majority, majority = majority)

        children = {} #child dictionary, since C4.5 doesn't only do binary splits and can split on numerous features 
        feature_values = np.unique(X[:, best_feature])

        for value in feature_values:
            idxs = np.where(X[:, best_feature] == value)[0]

            child_X = np.delete(X[idxs], best_feature, axis=1)
            child_y = y[idxs]

            children[value] = self.grow_tree(child_X, child_y, depth+1)

        return Node(feature = best_feature, children = children, majority = majority)


    def fit(self, X, y):
        """ Fit the data; Make the Decision tree
        Args:
            X (array-like): A 2D numpy array with the training data, excluding targets
            y (array-like): A 1D numpy array with the training targets
        Returns:
            self: this allows this to be chained, e.g. model.fit(X,y).predict(X_test)
        """
        # Replace unknown values :)
        X = np.array(X, dtype=object)
        y = np.array(y) #change from a pandas series to a numpy array
        
        X[X == None] = "Unknown" # Handle/change common unknown values if not already done
        X[X == "?"] = "Unknown" 

        self.root = self.grow_tree(X, y) # grow the tree, starting from the root 
        return self

    def traverse_tree(self, X, node):
        if node.is_leaf_node():
            return node.value

        feature_value = X[node.feature]

        if feature_value in node.children:
            # delete the feature for the child (since we can't split on the same feature twice), but keep x as 1D array
            new_x = np.delete(X, node.feature)
            return self.traverse_tree(new_x, node.children[feature_value])
        else:
            return node.majority # If there is no info, or an unknown feature, just return the majority class by default (as stored in node)

    def predict(self, X):
        X = np.array(X)  # ensure numpy
        # make sure X is 2D
        if X.ndim == 1:
            X = X.reshape(1, -1)
        predictions = np.array([self.traverse_tree(x, self.root) for x in X]) #for every value in x, traverse! output the final class predictions!
        return predictions

    def score(self, X, y):
        """ Return accuracy(Classification Acc) of model on a given dataset. Must implement own score function.

        Args:
            X (array-like): A 2D numpy array with data, excluding targets
            y (array-like): A 1D numpy array of the targets 
        """
        predictions = self.predict(X)
        y = np.array(y)  #change from a pandas series to a numpy array
        return np.mean(predictions == y)

In [ ]:
# Optional Debugging Dataset - Pizza Homework
pizza_dataset = np.array([[1,2,0],[0,0,0],[0,1,1],[1,1,1],[1,0,0],[1,0,1],[0,2,1],[1,0,0],[0,2,0]])
pizza_labels = np.array([2,0,1,2,1,2,1,1,0])

X = pizza_dataset
y = pizza_labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2)

my_clf = DTClassifier()
my_clf.fit(X_train, y_train)

print("Mini pizza dataset (to show that it is kind of working at least):")

y_pred = my_clf.predict(X_test)
print("\n\tWhat my Decision Tree Classifer predicts:")
print(f"\t{y_pred}")

score = my_clf.score(X_test, y_test)
print("\n\tThe Accuracy of my Decision Tree Classifier:")
print(f"\t{score}")

########################################################################
print("\n\nLarger Voting Dataset:")

# Get X and y
X = voting_na_df.drop(["Class"], axis = 1)
y = voting_na_df["Class"]

# One-hot encoding for nominal features:
X = pd.get_dummies(X)

# Do a split!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 123
)

# Dictionary for clean output
accuracies = {"Metric:" : ["Train Accuracy", "Test Accuracy"]}

# For each classifer, initialize, fit, predict, get accuracies
my_clf = DTClassifier()
my_clf.fit(X_train, np.ravel(y_train))
train_accuracy = my_clf.score(X_train, y_train)
test_accuracy = my_clf.score(X_test, np.ravel(y_test))
accuracies["My DTC"] = [train_accuracy, test_accuracy]

clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
train_accuracy = clf.score(X_train, y_train)
test_accuracy = clf.score(X_test, y_test)
accuracies["SKLearn's DTC"] = [train_accuracy, test_accuracy]

accuracies = pd.DataFrame(accuracies)
display(accuracies)

**Implementation notes**

The tree is built recursively (`grow_tree`): at each node, `find_best_feature` evaluates every available feature's gain ratio (computed by weighing information gain against a feature's intrinsic "split information," penalizing features with many unique values). The tree recurses on the resulting subsets until a node is pure, has no remaining features to split on, or no split improves gain ratio — at which point it becomes a leaf holding the majority class. Unknown or missing values are explicitly mapped to an `"Unknown"` category before training, so they're treated as their own informative attribute value rather than dropped or imputed. Prediction traverses the tree, falling back to a node's stored majority class if a test point's value wasn't seen during training for that feature.

One implementation detail worth calling out: this uses gain ratio (information gain normalized by split information) as its splitting criterion, distinguishing it from CART, which uses plain information gain (or Gini impurity) without that normalization — that distinction was actually a source of confusion partway through implementation, since the two are easy to conflate.

**Results**

On a small, hand-checkable "pizza" dataset, the classifier reached 100% test accuracy — a useful sanity check, though a small win given the very small dataset. More convincingly, on the full Voting dataset (with unknown attributes), this implementation's train and test accuracy matched scikit-learn's `DecisionTreeClassifier` exactly. That agreement is likely specific to this particular train/test split — a different split would probably reveal some divergence between the two algorithms — but it's a strong signal that the core algorithm is implemented correctly.

## Conclusion

Across every experiment here, the same underlying tension showed up repeatedly: unconstrained decision trees fit training data perfectly but often at the cost of generalization, while every method of controlling that (pre-pruning parameters, post-pruning via `ccp_alpha`, or simply limiting depth) trades some training accuracy for better-controlled complexity — usually a good trade, up to a point. The regression experiment was a useful reminder that hyperparameter tuning has limits: no amount of tuning fixed a dataset whose target variable simply had too wide a range for a single tree to model well.

Implementing C4.5 from scratch — and having it match scikit-learn's results on real data — was the most rewarding part of this project, and a good confirmation of understanding *why* decision trees make the splits they do, not just how to call the library that builds them.